In [ ]:
# Imports
from databricks.connect import DatabricksSession
from pyspark.sql import functions as F

In [ ]:
# Spark Session
spark = DatabricksSession.builder.serverless().profile("azure").getOrCreate()

In [ ]:
RUN_DATE = "2026-09-16"

sales_df_raw = (
    spark.read
    .option("header", True)
    .csv("abfss://bronze@learnenarb.dfs.core.windows.net/landing/sales/Sales.csv")
    .select(
        "*",
        F.col("_metadata.file_path").alias("_file_path"),
        F.col("_metadata.file_size").alias("_file_size"),
        F.col("_metadata.file_modification_time").alias("_last_modified_at"),
    )
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_run_date", F.lit(RUN_DATE).cast("date"))
)

sales_df_raw.show(5)

In [ ]:
sales_df_raw.printSchema()

In [ ]:
print("Rows: ", sales_df_raw.count())

In [ ]:
from src.bronze import ingest_bronze_autoloader

print(ingest_bronze_autoloader(spark, "electronics", "2026-09-17", "sales"))

In [ ]:
spark.stop()